# ConvexPi — Julia starter

Write a quant strategy in **Julia** and submit it. The grader runs your Julia `on_day` over a hidden
out-of-sample period and scores it with the *same* engine as Python/R. The `ConvexPi` package gives
you the market and one-call submit.

In [ ]:
# 1. Install the ConvexPi Julia package (market data + submit in one import).
using Pkg; Pkg.add(url = "https://github.com/convexpi/ConvexPi.jl")
using ConvexPi

In [ ]:
# 2. Load the exact market the grader uses (deterministic from the seed).
m = synthetic_market("train")      # fit on "train"; you're scored on the hidden "test"
prices = m.prices; features = m.features
println("prices: ", size(prices), " | features: ", join(keys(features), ", "))

## 3. Write your strategy

Define `on_day(day, features, prices, portfolio)` returning target weights (one per stock) — exactly
what the grader runs. Keep it as a string so you can both test it and submit it.

In [ ]:
strategy_code = """
function on_day(day, features, prices, portfolio)
    sig = copy(features["mom_1m"])     # 1-month momentum, z-scored across stocks
    sig[.!isfinite.(sig)] .= 0.0
    s = sum(abs.(sig))
    return s > 0 ? sig ./ s : sig      # long winners / short losers, gross leverage 1
end
"""
include_string(Main, strategy_code)    # define on_day locally

In [ ]:
# 4. (optional) sanity-check on one in-sample day
d = 300
w = on_day(d, Dict(n => vec(mat[d, :]) for (n, mat) in features), vec(prices[d, :]), zeros(size(prices, 2)))
println("weights sum to ", sum(w), " over ", length(w), " stocks")

## 5. Submit

Create an API key at **convexpi.ai/settings/api-keys** and set it below. Your OOS Sharpe appears on
the leaderboard within a few minutes.

In [ ]:
ENV["CONVEXPI_API_KEY"] = "cpk_..."   # <- your key
submit("my-julia-momentum", strategy_code)   # slug defaults to "demo-fall-2026"